<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.en/cap05/cap05.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

# Chapter 5 - Exercises Proposed (EPs)

**Objective:** Apply the concepts presented in Chapter 5 by solving the proposed exercises.

---

## Instructions

1. Read the statements of each exercise carefully.
2. Implement the solutions in the provided code cells.
3. Run the tests to verify your answers.
4. When finished, save and submit the notebook as directed by your instructor.

---

## Exercise 1

**Statement:** Write a program that reads an RGB image and converts it to grayscale using the weighted average method. Display the original and converted images side by side.

**Hint:** Use `cv2.cvtColor` with `cv2.COLOR_BGR2GRAY` or implement the formula manually.

---

## Exercise 2

**Statement:** Implement a function to apply a **Gaussian blur** to an image. Compare the result with the original using different kernel sizes (e.g., 3x3, 5x5, 7x7).

**Hint:** Use `cv2.GaussianBlur` with adjustable `ksize` and `sigmaX`.

---

## Exercise 3

**Statement:** Perform edge detection using the **Canny** algorithm. Adjust the thresholds to observe their effect on the detected edges.

**Hint:** Use `cv2.Canny` with `threshold1` and `threshold2` parameters.

---

## Exercise 4

**Statement:** Use morphological operations (erosion and dilation) to remove noise from a binary image. Show the results after each operation.

**Hint:** Use `cv2.erode` and `cv2.dilate` with a structuring element (e.g., `cv2.getStructuringElement`).

---

## Exercise 5

**Statement:** Develop a simple **image segmentation** algorithm by thresholding (Otsu's method). Display the histogram and the resulting binary image.

**Hint:** Use `cv2.threshold` with `cv2.THRESH_BINARY + cv2.THRESH_OTSU`.

---

## Submission

After completing all exercises, ensure that the notebook runs without errors. Include comments in your code explaining the main steps. Submit the file according to your course's guidelines.

---

**End of Chapter 5 Proposed Exercises.**

## 💻 **Hands-On Section with Programming Exercises**

🚧 **Under construction!**

The present list of programming exercises (PE) consolidates the theoretical formulations presented throughout Chapter 5 — Transforms and Compression — through a practical applied track. The exercises are structured around matrices of reduced dimensions, enabling analytical validation and manual inspection of each coefficient, while maintaining the methodological consistency adopted in previous chapters.

The sequencing of the exercises rigorously reproduces the conceptual flow of the chapter: it begins with the explicit implementation of the Discrete Fourier Transform (DFT) from its fundamental mathematical definition; it proceeds to the design of low-pass filters and *notch* masks in the frequency domain; it applies coefficient quantization (the core of lossy compression); and it concludes with the integration of these steps in the construction of a simplified JPEG compression *pipeline* and the perceptual analysis of image formats.

> ### ❗ Guidelines for Solving the Programming Exercises
>
> In all exercises in this chapter, the coordinates of the **spectrum center** (the origin of spatial frequencies after applying the `fftshift` displacement) must be determined via integer division. For a matrix with $L$ rows and $C$ columns, the zero-frequency component is located at the position:
>
> $$
> (c_y, c_x) = \left( \left\lfloor \frac{L}{2} \right\rfloor, \left\lfloor \frac{C}{2} \right\rfloor \right)
> $$
>
> This convention is strictly identical to that adopted by the `np.fft.fftshift` function. Furthermore, in all steps requiring discretization or numerical rounding (whether in the quantization of AC coefficients or in the final reconstruction of pixels), standard rounding to the nearest integer (*round half away from zero*) must be employed, mitigating ambiguities in values with a fraction exactly equal to $0.5$.

### 🎯 Objective of this Notebook

The notebook enables the development, validation, organization, and testing of solutions for **Programming Exercises (PEs)** in interactive environments, such as Colab, using the same test cases as Moodle, copying them there only when recording the official grade.

#### *Download*

Download `morph.py` and `testsuite.py` by running the cell below:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Running the Tests
To evaluate the tests, run `TestSuite("EP05_01.extensão").run()` in a new cell, replacing the extension with the one used for your language (`.py`, `.java`, `.c`, `.cpp`, `.js`, or `.r`). The system downloads the test cases from GitHub, runs the program, and calculates the grade automatically.

To test Python code directly, without saving a file, use `run_code(codigo)` passing the code as a *string* in a variable named `codigo`:

```python
codigo = """
from morph import mm
# ... your code here ...
"""
TestSuite("EP05_01").run_code(codigo)
```

### EP05_01 🟢 Ideal Low-Pass Filter by Distance in the Spectrum

In an **old document scanner**, the sensor captures crumpled paper and fiber texture along with the text — high-frequency noise that "pollutes" the spectrum at the edges. The maintenance technician has no access to the original image, only to the **magnitude spectrum already computed** by the scanner's software. Their job is simple and surgical: keep only the **central circle** of low frequencies (the global structure of the document) and erase everything outside the radius $D_0$, eliminating the fine texture without even needing to touch the spatial image.

This is the **Ideal Low-Pass Filter (LPFI)**: the most direct spectral operation in the chapter, but also the one that best reveals the anatomy of a centered spectrum.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns) from the magnitude spectrum — already provided **centered** (equivalent to the output of `np.fft.fftshift`).
2. **Cutoff frequency:** Read the integer $D_0$.
3. **Data:** Read the integer values of the magnitude matrix, row by row.
4. **Spectrum center:** Compute $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
5. **Distance:** For each position $(u,v)$, compute
$$
D(u,v) = \sqrt{(u-c_y)^2 + (v-c_x)^2}
$$
6. **Ideal mask:** Apply
$$
H(u,v) = \begin{cases} 1, & D(u,v) \le D_0 \\ 0, & D(u,v) > D_0 \end{cases}
$$
7. **Filtering:** The output value is $\text{mag}'(u,v) = \text{mag}(u,v) \cdot H(u,v)$.
8. **Output:** Display the filtered matrix with dimensions $L \times C$.

#### 📌 Computational Constraints

* **Non-strict comparison:** the criterion uses $D(u,v) \le D_0$ (the boundary belongs to the filter, i.e., it is kept).
* **Type:** all input and output values are integers; the distance is computed in floating point only internally.
* **No magnitude rounding:** since the input is already integer and the mask is binary (0 or 1), the output never requires rounding.

#### 🧠 Theoretical Background

| Region | Distance to center | Filter effect |
|---|---|---|
| **Center** ($D \le D_0$) | Low frequencies | Preserved — global structure maintained |
| **Edges** ($D > D_0$) | High frequencies | Zeroed — texture and noise removed |
| **Small $D_0$** | — | Reconstructed image would be very blurry |
| **Large $D_0$** | — | Little filtering; almost all energy preserved |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Line 3: Integer $D_0$.
* Following lines: Integer elements of the magnitude matrix (centered).

**Output:**

* Filtered matrix with $L$ rows and $C$ columns, separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 3<br>3<br>1<br>10 20 30<br>40 50 60<br>70 80 90 | 0 20 0<br>40 50 60<br>0 80 0 | Center $(1,1)$. Corners have $D=\sqrt{2}\approx1.41 > 1$, hence they are zeroed; orthogonal neighbors have $D=1 \le 1$ and are kept. |
| 1<br>3<br>0<br>5 9 7 | 0 9 0 | $L=1, C=3$: center at $(0,1)$. Only the central position itself ($D=0$) survives $D_0=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0501" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0501 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0501 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0501 button:hover { background: #e8dfcf; }
  #sim-ep0501 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0501_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0501_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0501_grid_stats { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 12px; }
  .sim-ep0501_stat_box { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 10px; padding: 10px; text-align: center; }
  .sim-ep0501_stat_label { font-size: 9.5px; color: #8a8371; text-transform: uppercase; letter-spacing: 0.04em; margin-bottom: 2px; font-weight: 700; }
  .sim-ep0501_stat_value { font-size: 16px; font-weight: 700; font-family: monospace; color: #26241d; }
  .sim-ep0501_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP05_01: Ideal Low-Pass Filter</span>
  <span class="sim-ep0501_pill">H = (D &le; D₀) ? 1 : 0</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0501_panel" style="margin-bottom:14px;">
    
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Cutoff radius (D₀): <span id="sim-ep0501_vl_d0" style="font-family:monospace; color:#26241d;">1</span>
      </label>
    </div>
    
    <input id="sim-ep0501_sl_d0" type="range" min="0" max="4" step="1" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">Adjust D₀ and observe which positions of the 5&times;5 spectrum survive the filter.</div>

  </div>

  <!-- Exibição das Grades de Espectro -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">Original Spectrum (Magnitude)</div>
      <div id="sim-ep0501_grid_orig" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0501_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">Filtered Result</div>
      <div id="sim-ep0501_grid_new" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0501_debug" class="sim-ep0501_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    –
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep01(root){
    if (!root || root.dataset.sim05Ep01Init) return;
    root.dataset.sim05Ep01Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(10 * (i + 1) + j + 1);
      }
      mag.push(row);
    }

    var d0el = root.querySelector('#sim-ep0501_sl_d0');
    var d0v  = root.querySelector('#sim-ep0501_vl_d0');
    var go   = root.querySelector('#sim-ep0501_grid_orig');
    var gn   = root.querySelector('#sim-ep0501_grid_new');
    var dbg  = root.querySelector('#sim-ep0501_debug');

    function cellStyle(active){
      if (active) {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      } else {
        return 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      }
    }

    function render(){
      var D0 = parseInt(d0el.value, 10);
      d0v.textContent = D0;
      go.innerHTML = '';
      gn.innerHTML = '';
      var kept = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d = Math.sqrt((i - cy) * (i - cy) + (j - cx) * (j - cx));
          var keep = d <= D0;
          if (keep) kept++;

          var co = document.createElement('div');
          co.className = 'sim-ep0501_cell';
          co.style.cssText = cellStyle(true);
          co.textContent = mag[i][j];
          go.appendChild(co);

          var cn = document.createElement('div');
          cn.className = 'sim-ep0501_cell';
          cn.style.cssText = cellStyle(keep);
          cn.textContent = keep ? mag[i][j] : 0;
          gn.appendChild(cn);
        }
      }

      dbg.textContent = 'Center = (' + cy + ', ' + cx + ')  |  D₀ = ' + D0 + '  |  Coeficientes mantidos: ' + kept + ' / ' + (N * N);
    }

    d0el.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep01(){
    var root = document.getElementById('sim-ep0501');
    if (root) initSim05Ep01(root); else setTimeout(tryInitSim05Ep01, 200);
  }
  tryInitSim05Ep01();
})();
</script>
""")

**Figure 5.1:** EP05_01 Simulator: Ideal Low-Pass Filter in the Spectrum


In [ ]:
%%writefile EP05_01.py
# Python code

In [ ]:
TestSuite("EP05_01.py").run()

### EP05_02 🟡 *Notch* Filter: Removing Periodic Peaks

An **industrial inspection** camera captures images of circuit boards, but the production line's power supply introduces a **periodic electrical interference** — a stripe pattern almost imperceptible to the naked eye, yet visible in the Fourier spectrum as **pairs of bright peaks** symmetrically positioned around the center. The computer vision team cannot redo the capture: they must **surgically locate and erase** these peak pairs in the spectrum, preserving all other useful image information.

This is the role of the **notch reject filter**: unlike a low-pass filter (which affects a continuous region), it targets **specific points and their symmetric counterparts**, leaving the rest of the spectrum untouched.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $L$ (rows) and $C$ (columns) of the centered magnitude spectrum.
2. **Data:** Read the integer values of the magnitude matrix, row by row.
3. **Peaks:** Read the integer $K$ (number of peak pairs to remove).
4. **For each of the $K$ peaks:** read three integers $\Delta v$, $\Delta u$, $r$ — vertical offset, horizontal offset, and notch radius.
5. **Spectrum center:** $(c_y, c_x) = (L \mathbin{//} 2,\; C \mathbin{//} 2)$.
6. **Symmetric suppression:** for each peak, zero out **all** positions $(u,v)$ such that the distance to the point $(c_y+\Delta v,\, c_x+\Delta u)$ is $\le r$, **and also** all positions with distance $\le r$ to the symmetric point $(c_y-\Delta v,\, c_x-\Delta u)$.
7. **Output:** Display the resulting matrix with dimensions $L \times C$.

#### 📌 Computational Constraints

* **Mandatory symmetry:** each reported peak generates **two** zeroed disks (the point and its symmetric counterpart relative to the center) — forgetting the symmetric point is the most common mistake.
* **Overlap:** if two disks overlap, the position remains zeroed (there is no "addition" or restoration).
* **Non-strict comparison:** a position is zeroed if $\text{distance} \le r$.
* **Reading order:** the $K$ peaks must be processed in the order they appear in the input, but the final result is independent of order (zeroing operations are commutative).

#### 🧠 Theoretical Foundation

| Concept | Role in the notch filter |
|---|---|
| **Peak at $(\Delta v, \Delta u)$** | Frequency of the periodic interference visually detected in the spectrum |
| **Symmetric point $(-\Delta v,-\Delta u)$** | Every DFT of a real signal is Hermitian: peaks always appear in pairs symmetric about the center |
| **Radius $r$** | Controls the "width" of rejection — a large $r$ removes more energy around the peak, but also useful information |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $L$.
* Line 2: Integer $C$.
* Following lines: Integer elements of the magnitude matrix (centered), $L$ lines.
* Next line: Integer $K$.
* Following $K$ lines: three integers $\Delta v$, $\Delta u$, $r$ (space-separated).

**Output:**

* The resulting matrix in $L$ lines and $C$ columns, space-separated.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 5<br>5<br>1 2 3 4 5<br>6 7 8 9 10<br>11 12 13 14 15<br>16 17 18 19 20<br>21 22 23 24 25<br>1<br>1 1 0 | 1 2 3 4 5<br>6 0 8 9 10<br>11 12 13 14 15<br>16 17 18 0 20<br>21 22 23 24 25 | Center $(c_y, c_x) = (2, 2)$. Reported peak $(\Delta v, \Delta u) = (1, 1)$ generates the point $(3, 3)$ (value 19) and its symmetric counterpart $(1, 1)$ (value 7), both zeroed with $r=0$ (only the exact points). |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0502" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0502 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0502 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0502 button:hover { background: #e8dfcf; }
  #sim-ep0502 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0502_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0502_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0502_grid_ctrls { display: grid; grid-template-columns: repeat(auto-fit, minmax(130px, 1fr)); gap: 12px; }
  .sim-ep0502_cell { width: 42px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP05_02: Notch Filter</span>
  <span class="sim-ep0502_pill">Symmetric Pair</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0502_panel" style="margin-bottom:14px;">
    <div class="sim-ep0502_grid_ctrls">
      
      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;v</label>
          <span id="sim-ep0502_vl_dv" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_dv" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">&Delta;u</label>
          <span id="sim-ep0502_vl_du" style="font-family:monospace; font-weight:700; color:#26241d;">1</span>
        </div>
        <input id="sim-ep0502_sl_du" type="range" min="-2" max="2" step="1" value="1">
      </div>

      <div>
        <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:4px;">
          <label style="font-size:11px; font-weight:700; color:#5e5a4a;">Radius (r)</label>
          <span id="sim-ep0502_vl_r" style="font-family:monospace; font-weight:700; color:#26241d;">0</span>
        </div>
        <input id="sim-ep0502_sl_r" type="range" min="0" max="2" step="1" value="0">
      </div>

    </div>

    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:10px; text-align:center;">
      Move &Delta;v e &Delta;u to choose the peak &mdash; note that the symmetric pair is also filtered.
    </div>
  </div>

  <!-- Espectro 5x5 -->
  <div class="sim-ep0502_panel" style="text-align:center; margin-bottom:14px;">
    <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
      Spectrum 5&times;5 (Red = Removed by the Filter)
    </div>
    <div id="sim-ep0502_grid" style="display:grid; grid-template-columns:repeat(5, 42px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0502_debug" class="sim-ep0502_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep02(root){
    if (!root || root.dataset.sim05Ep02Init) return;
    root.dataset.sim05Ep02Init = "1";

    var N = 5, cy = Math.floor(N / 2), cx = Math.floor(N / 2);
    var mag = [];
    for (var i = 0; i < N; i++){
      var row = [];
      for (var j = 0; j < N; j++){
        row.push(i * 5 + j + 1);
      }
      mag.push(row);
    }

    var dv  = root.querySelector('#sim-ep0502_sl_dv');
    var du  = root.querySelector('#sim-ep0502_sl_du');
    var r   = root.querySelector('#sim-ep0502_sl_r');
    var dvv = root.querySelector('#sim-ep0502_vl_dv');
    var duv = root.querySelector('#sim-ep0502_vl_du');
    var rv  = root.querySelector('#sim-ep0502_vl_r');

    var grid = root.querySelector('#sim-ep0502_grid');
    var dbg  = root.querySelector('#sim-ep0502_debug');

    function cellStyle(kill){
      if (kill) {
        return 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
      } else {
        return 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
    }

    function render(){
      var DV = parseInt(dv.value, 10);
      var DU = parseInt(du.value, 10);
      var R  = parseInt(r.value, 10);

      dvv.textContent = DV;
      duv.textContent = DU;
      rv.textContent  = R;

      var p1 = [cy + DV, cx + DU];
      var p2 = [cy - DV, cx - DU];

      grid.innerHTML = '';
      var removed = 0;

      for (var i = 0; i < N; i++){
        for (var j = 0; j < N; j++){
          var d1 = Math.sqrt((i - p1[0]) * (i - p1[0]) + (j - p1[1]) * (j - p1[1]));
          var d2 = Math.sqrt((i - p2[0]) * (i - p2[0]) + (j - p2[1]) * (j - p2[1]));
          var kill = (d1 <= R) || (d2 <= R);

          if (kill) removed++;

          var c = document.createElement('div');
          c.className = 'sim-ep0502_cell';
          c.style.cssText = cellStyle(kill);
          c.textContent = kill ? 0 : mag[i][j];
          grid.appendChild(c);
        }
      }

      dbg.textContent = 'Center = (' + cy + ', ' + cx + ')  |  Pico = (' + p1[0] + ', ' + p1[1] + ')  |  Simétrico = (' + p2[0] + ', ' + p2[1] + ')  |  Removidos: ' + removed;
    }

    [dv, du, r].forEach(function(el){
      el.addEventListener('input', render);
    });

    render();
  }

  function tryInitSim05Ep02(){
    var root = document.getElementById('sim-ep0502');
    if (root) initSim05Ep02(root); else setTimeout(tryInitSim05Ep02, 200);
  }
  tryInitSim05Ep02();
})();
</script>
""")

**Figure 5.2:** EP05_02 Simulator: Notch Filter


In [ ]:
%%writefile EP05_02.py
# Python code

In [ ]:
TestSuite("EP05_02.py").run()

### EP05_03 🟠 DCT Quantization: The True Source of Compression

A **photo gallery** application needs to reduce the size of thousands of images before *uploading* them to the cloud, without recoding everything from scratch. The engineer in charge already has the **DCT coefficients** for each $4\times4$ block calculated (the computationally expensive step has already been done) — only the **quantization table** needs to be applied, the step that actually discards information and generates compression. High-frequency coefficients, which are less perceptible to the human eye, receive large divisors and tend to become **zero**; low-frequency coefficients, which are more perceptible, receive small divisors and survive almost intact.

You will implement exactly this step: **quantize and dequantize** (divide, round, multiply back) — the heart of JPEG *lossy* compression.

#### 📋 Implementation Guidelines

1. **Block size:** Read the integer $N$ ($N \times N$ block).
2. **Coefficients:** Read the matrix $C$ of DCT coefficients, $N$ rows with $N$ integers each (they may be negative).
3. **Quantization table:** Read the matrix $Q$, $N$ rows with $N$ positive integers each.
4. **Quantization:** For each position $(u,v)$, compute the quantized index
$$
\tilde{C}(u,v) = \text{round}\!\left(\frac{C(u,v)}{Q(u,v)}\right)
$$
using standard rounding to the nearest integer (intermediate `.5` values never occur in the test cases).
5. **Dequantization (reconstruction):** Compute
$$
C'(u,v) = \tilde{C}(u,v) \times Q(u,v)
$$
6. **Output:** Display the reconstructed matrix $C'$, $N \times N$, integers.

#### 📌 Computational Constraints

* **Complete *round-trip*:** The output is the **reconstructed** coefficient ($\tilde{C} \times Q$), not the isolated quantized index.
* **Floating-point division:** The division $C(u,v)/Q(u,v)$ must be performed in floating point before rounding — truncated integer division will produce an incorrect result.
* **Preserved sign:** Negative coefficients retain their sign after quantization and reconstruction.
* **$Q(u,v) > 0$ always:** There is no need to handle division by zero.

#### 🧠 Theoretical Background

| Coefficient | Frequency | Typical value of $Q$ | Effect of quantization |
|---|---|---|---|
| $C(0,0)$ | DC (block average) | Small | Almost always survives — dominates energy |
| $C(u,v)$ low $u+v$ | Low frequency | Small/medium | Partially preserved |
| $C(u,v)$ high $u+v$ | High frequency | Large | Often becomes zero — source of compression |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $N$.
* The next $N$ lines: matrix $C$ (DCT coefficients, integers, may be negative).
* The next $N$ lines: matrix $Q$ (quantization table, positive integers).

**Output:**

* Reconstructed matrix $C'$, $N \times N$, integers separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4<br>50 10 -5 0<br>8 -3 2 1<br>0 1 0 0<br>2 0 0 -1<br>2 5 7 8<br>4 7 8 11<br>6 8 11 12<br>9 11 12 14 | 50 10 -7 0<br>8 0 0 0<br>0 0 0 0<br>0 0 0 0 | $C(0,0)=50/2=25 \to 25\times2=50$ (preserved). $C(0,2)=-5/7\approx-0.71\to-1\to-1\times7=-7$. Meanwhile, $C(1,1)=-3/7\approx-0.43\to0$: zeroed by quantization — most of the block becomes zero, illustrating the energy compaction in the upper-left corner. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0503" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0503 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0503 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0503 button:hover { background: #e8dfcf; }
  #sim-ep0503 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0503_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0503_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0503_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP05_03: DCT Quantization</span>
  <span class="sim-ep0503_pill">round(C / Q) &times; Q</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0503_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Q Scale (Aggressiveness): <span id="sim-ep0503_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0503_sl_s" type="range" min="0.25" max="4" step="0.25" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the Q scale and see how many coefficients survive (non-zero) after the round-trip.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        DCT Coefficients (C)
      </div>
      <div id="sim-ep0503_grid_c" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0503_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstructed (round(C / Q) &middot; Q)
      </div>
      <div id="sim-ep0503_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0503_debug" class="sim-ep0503_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep03(root){
    if (!root || root.dataset.sim05Ep03Init) return;
    root.dataset.sim05Ep03Init = "1";

    var C = [[50, 10, -5, 0], [8, -3, 2, 1], [0, 1, 0, 0], [2, 0, 0, -1]];
    var Qbase = [[2, 5, 7, 8], [4, 7, 8, 11], [6, 8, 11, 12], [9, 11, 12, 14]];

    var s   = root.querySelector('#sim-ep0503_sl_s');
    var sv  = root.querySelector('#sim-ep0503_vl_s');
    var gc  = root.querySelector('#sim-ep0503_grid_c');
    var gr  = root.querySelector('#sim-ep0503_grid_r');
    var dbg = root.querySelector('#sim-ep0503_debug');

    function cell(v, faded){
      var c = document.createElement('div');
      c.className = 'sim-ep0503_cell';
      if (faded) {
        c.style.cssText = 'background:#fafaf7; color:#8a8371; border:1px solid #e4dcc8;';
      } else {
        c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      }
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      gc.innerHTML = '';
      gr.innerHTML = '';
      var zeros = 0, total = 16;

      for (var i = 0; i < 4; i++){
        for (var j = 0; j < 4; j++){
          gc.appendChild(cell(C[i][j], false));
          var Q = Qbase[i][j] * scale;
          var q = Math.round(C[i][j] / Q);
          var rec = Math.round(q * Q);
          if (rec === 0) zeros++;
          gr.appendChild(cell(rec, rec === 0));
        }
      }

      dbg.textContent = 'Zeros: ' + zeros + ' / ' + total + '  |  Quanto maior a escala de Q, mais zeros — maior compressão, menor qualidade.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep03(){
    var root = document.getElementById('sim-ep0503');
    if (root) initSim05Ep03(root); else setTimeout(tryInitSim05Ep03, 200);
  }
  tryInitSim05Ep03();
})();
</script>
""")

**Figure 5.3:** EP05_03 Simulator: DCT Quantization (*round-trip*)


In [ ]:
%%writefile EP05_03.py
# Python code

In [ ]:
TestSuite("EP05_03.py").run()

### EP05_04 🔴 Implementing the 2D DFT from the Definition

A research laboratory in **computational astronomy** received, from an old mission, a small experimental sensor whose raw data cannot be processed by modern FFT libraries — the validation environment is isolated and only allows basic arithmetic operations. The team needs to **reimplement the 2D Discrete Fourier Transform from the mathematical definition itself**, cell by cell, to later compare bit by bit with `np.fft.fft2` in another environment.

This is the most conceptual exercise on the list: there are no shortcuts. You will implement the double summation from [Equation 5](#eq-05-dft) directly, demonstrating *why* the FFT exists — and the computational cost it avoids.

#### 📋 Implementation Guidelines

1. **Dimensions:** Read the integers $M$ (rows) and $N$ (columns) of the image $f(x,y)$.
2. **Data:** Read the integer values of $f(x,y)$, row by row.
3. **2D DFT:** For each frequency pair $(u,v)$ with $u=0,\ldots,M-1$ and $v=0,\ldots,N-1$, compute
$$
F(u,v) = \sum_{x=0}^{M-1}\sum_{y=0}^{N-1} f(x,y)\, e^{-j2\pi\left(\frac{ux}{M}+\frac{vy}{N}\right)}
$$
using Euler's identity $e^{-j\theta} = \cos(\theta) - j\sin(\theta)$ to separate the real and imaginary parts — **do not use any ready-made FFT function**.
4. **Magnitude:** Compute $|F(u,v)| = \sqrt{\text{Re}(F)^2 + \text{Im}(F)^2}$ and round to the nearest integer.
5. **Output:** Display the matrix of rounded magnitudes, $M \times N$, in the same order (without `fftshift` — the DC remains at $(0,0)$).

#### 📌 Computational Constraints

* **Using FFT libraries is prohibited:** the implementation must compute the double summations explicitly (nested loops), even if slower.
* **No `fftshift`:** the output maintains the raw DFT convention, with the DC component at $F(0,0)$ (upper-left corner).
* **Rounding:** the final magnitude must be rounded to the nearest integer; in the test cases there is no `.5` ambiguity.
* **Precision:** small floating-point errors (on the order of $10^{-6}$) before rounding are expected and do not affect the final integer result.

#### 🧠 Theoretical Foundation

| Element | Meaning |
|---|---|
| $F(0,0)$ | DC component — sum of all pixels, $F(0,0) = \sum f(x,y)$ |
| Real part $\text{Re}(F)$ | Projection of the signal onto cosines |
| Imaginary part $\text{Im}(F)$ | Projection of the signal onto sines |
| Complexity of this implementation | $\mathcal{O}((MN)^2)$ — this is why the FFT, with $\mathcal{O}(MN\log(MN))$, is indispensable for real images |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $M$.
* Line 2: Integer $N$.
* Following lines: Integer elements of $f(x,y)$, $M$ lines.

**Output:**

* Matrix of rounded magnitudes $|F(u,v)|$, $M \times N$, separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 2<br>2<br>1 2<br>3 4 | 10 2<br>4 0 | $F(0,0)=1+2+3+4=10$ (DC = total sum). $F(0,1)=(1-2)+(3-4)=-2 \to |F|=2$. $F(1,0)=(1+2)-(3+4)=-4\to|F|=4$. $F(1,1)=(1-2)-(3-4)=0$. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0504" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0504 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0504 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0504 button:hover { background: #e8dfcf; }
  .sim-ep0504_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0504_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0504_cell { width: 52px; height: 42px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 13px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP05_04: 2D DFT &mdash; Direct Definition</span>
  <span class="sim-ep0504_pill">&Sigma;&Sigma; f(x,y) e<sup>-j2&pi;(&hellip;)</sup></span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Instrução -->
  <div class="sim-ep0504_panel" style="margin-bottom:14px; text-align:center;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600;">
      Click on the f(x,y) cells to change the values (click increments +1; Shift + click decrements -1) and watch |F(u,v)| recalculated live.
    </div>
  </div>

  <!-- Exibição das Grades 2x2 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        f(x,y) &mdash; Spatial Domain
      </div>
      <div id="sim-ep0504_grid_f" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0504_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        |F(u,v)| &mdash; Magnitude (No Shift)
      </div>
      <div id="sim-ep0504_grid_F" style="display:grid; grid-template-columns:repeat(2, 52px); gap:6px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0504_debug" class="sim-ep0504_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep04(root){
    if (!root || root.dataset.sim05Ep04Init) return;
    root.dataset.sim05Ep04Init = "1";

    var f = [[1, 2], [3, 4]];
    var gf  = root.querySelector('#sim-ep0504_grid_f');
    var gF  = root.querySelector('#sim-ep0504_grid_F');
    var dbg = root.querySelector('#sim-ep0504_debug');

    function render(){
      gf.innerHTML = '';
      gF.innerHTML = '';

      for (var x = 0; x < 2; x++){
        for (var y = 0; y < 2; y++){
          (function(xx, yy){
            var c = document.createElement('div');
            c.className = 'sim-ep0504_cell';
            c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7; cursor:pointer;';
            c.textContent = f[xx][yy];
            c.addEventListener('click', function(e){
              if (e.shiftKey){ f[xx][yy]--; } else { f[xx][yy]++; }
              render();
            });
            gf.appendChild(c);
          })(x, y);
        }
      }

      var M = 2, N = 2;
      for (var u = 0; u < M; u++){
        for (var v = 0; v < N; v++){
          var re = 0, im = 0;
          for (var x = 0; x < M; x++){
            for (var y = 0; y < N; y++){
              var theta = 2 * Math.PI * (u * x / M + v * y / N);
              re += f[x][y] * Math.cos(theta);
              im -= f[x][y] * Math.sin(theta);
            }
          }
          var mag = Math.round(Math.sqrt(re * re + im * im));
          var c = document.createElement('div');
          c.className = 'sim-ep0504_cell';
          c.style.cssText = 'background:#fdecea; color:#c0392b; border:1px solid #f5b7b1;';
          c.textContent = mag;
          gF.appendChild(c);
        }
      }

      dbg.textContent = 'F(0,0) = sum of all pixels = ' + (f[0][0] + f[0][1] + f[1][0] + f[1][1]) + ' (componente DC)';
    }

    render();
  }

  function tryInitSim05Ep04(){
    var root = document.getElementById('sim-ep0504');
    if (root) initSim05Ep04(root); else setTimeout(tryInitSim05Ep04, 200);
  }
  tryInitSim05Ep04();
})();
</script>
""")

**Figure 5.4:** EP05_04 Simulator: Manual 2D DFT


In [ ]:
%%writefile EP05_04.py
# Python code

In [ ]:
TestSuite("EP05_04.py").run()

### EP05_05 🏆 Complete JPEG Pipeline: DCT, Quantization, and Reconstruction

You have been hired to create, from scratch, a **didactic JPEG codec** in an embedded environment, without any available image library—only basic mathematical operations. The client wants to understand exactly where quality is lost and where it is recovered, block by block. This is the final challenge of the chapter: integrate **everything** that has been studied—the orthonormal DCT-II, perceptual quantization, and IDCT-based reconstruction—into a single end-to-end *pipeline*, processing an $N \times N$ block from start to finish, exactly as the JPEG standard does internally, $8\times8$ pixels at a time.

#### 📋 Implementation Guidelines

1. **Block dimension:** Read the integer $N$.
2. **Original block:** Read the pixel matrix $f(x,y)$, $N$ rows with $N$ integers in $[0,255]$.
3. **Quantization table:** Read the matrix $Q$, $N \times N$ positive integers.
4. **Centering:** Subtract 128 from each pixel: $g(x,y) = f(x,y) - 128$.
5. **Orthonormal 2D DCT-II:** Compute
$$
C(u,v) = \alpha(u)\,\alpha(v)\sum_{x=0}^{N-1}\sum_{y=0}^{N-1} g(x,y)\,\cos\!\left[\frac{\pi(2x+1)u}{2N}\right]\cos\!\left[\frac{\pi(2y+1)v}{2N}\right]
$$
with $\alpha(0)=\sqrt{1/N}$ and $\alpha(k)=\sqrt{2/N}$ for $k>0$.
6. **Quantization:** $\tilde{C}(u,v) = \text{round}(C(u,v)/Q(u,v))$.
7. **Dequantization:** $C'(u,v) = \tilde{C}(u,v)\times Q(u,v)$.
8. **2D IDCT-II (orthonormal inverse):** Compute $g'(x,y)$ from $C'(u,v)$ using the corresponding inverse transform (same basis, summation over $u,v$).
9. **Centering reversal and rounding:** $f'(x,y) = \text{round}(g'(x,y) + 128)$, restricted to the interval $[0,255]$ (*clipping*).
10. **Output:** Display the reconstructed block $f'$, $N \times N$, integers.

#### 📌 Computational Constraints

* ***Complete pipeline mandatory:** all six stages (center, DCT, quantize, dequantize, IDCT, revert) must be implemented—skipping quantization will not pass the tests, as the result would be identical to the original.
* ***Clipping:** reconstructed values outside $[0,255]$ must be truncated (0 if negative, 255 if greater than 255).
* **Rounding:** both in quantization and in final pixel reconstruction, use standard rounding; the test cases avoid `.5` ambiguity.
* **Orthonormal basis:** the normalization $\alpha(u)$ and $\alpha(v)$ must be applied exactly as specified—without it, the IDCT will not reconstruct correctly.

#### 🧠 Theoretical Foundation

| Stage | Analogous in the real JPEG standard | Where quality is lost |
|---|---|---|
| Centering | Same—DCT assumes a signal centered at zero | No loss |
| DCT-II | Steps 3–4 of the *pipeline* ([Table 5](#tbl-05-pipeline-jpeg)) | No loss (exact and reversible transformation) |
| Quantization | Step 5—division by $Q(u,v)$ | **Main source of loss**—high-frequency coefficients become zero |
| IDCT | Final reconstruction | Reconstructs exactly the *quantized* coefficients, not the original ones |

#### 📦 Input and Output Specification (VPL)

**Input:**

* Line 1: Integer $N$.
* Next $N$ lines: original block $f(x,y)$, integers in $[0,255]$.
* Next $N$ lines: quantization table $Q$, positive integers.

**Output:**

* Reconstructed block $f'(x,y)$, $N \times N$, integers in $[0,255]$, separated by spaces.

#### 📌 Examples

| Input | Output | Observation |
|---|---|---|
| 4<br>120 130 125 128<br>115 140 135 122<br>118 150 160 130<br>110 120 145 138<br>4 6 8 10<br>6 8 10 12<br>8 10 12 16<br>10 12 16 20 | 118 126 119 131<br>114 143 140 119<br>117 149 159 130<br>107 121 146 139 | After DCT, aggressive quantization of high frequencies (large $Q$ values in the lower-right corner) and IDCT-based reconstruction, the block remains **close** to the original but not identical—the difference is the cost of *lossy* compression. |

#### 💡 Debugging Tip

If the result does not match, check in this order: (1) the raw DCT coefficients (before quantization)—they should reconstruct the original **exactly** via IDCT if you skip steps 6–7; (2) the $\alpha(u)$ table—a common mistake is applying $\sqrt{2/N}$ also for $u=0$; (3) the quantization rounding, which must occur **before** multiplying back by $Q$.

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0505" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0505 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0505 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0505 button:hover { background: #e8dfcf; }
  #sim-ep0505 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0505_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0505_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
  .sim-ep0505_cell { width: 46px; height: 36px; display: flex; align-items: center; justify-content: center; border-radius: 6px; font-size: 11px; font-weight: 700; font-family: monospace; transition: all 0.15s ease; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulator EP05_05: JPEG Pipeline (Block 4&times;4)</span>
  <span class="sim-ep0505_pill">DCT &rarr; Q &rarr; IDCT</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0505_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Q scale (1 = Base Table, Larger = More Loss): <span id="sim-ep0505_vl_s" style="font-family:monospace; color:#26241d;">1.00&times;</span>
      </label>
    </div>
    
    <input id="sim-ep0505_sl_s" type="range" min="0.5" max="5" step="0.5" value="1">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Adjust the quantization scale factor and watch the reconstructed block move away from (or closer to) the original.
    </div>
  </div>

  <!-- Exibição das Grades 4x4 -->
  <div style="display:grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap:14px; margin-bottom:14px;">
    
    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Original Block
      </div>
      <div id="sim-ep0505_grid_o" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

    <div class="sim-ep0505_panel" style="text-align:center;">
      <div style="font-size:10px; font-weight:700; color:#8a8371; text-transform:uppercase; margin-bottom:12px; letter-spacing:0.04em;">
        Reconstructed (DCT &rarr; Q &rarr; IDCT)
      </div>
      <div id="sim-ep0505_grid_r" style="display:grid; grid-template-columns:repeat(4, 46px); gap:4px; justify-content:center;"></div>
    </div>

  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0505_debug" class="sim-ep0505_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim05Ep05(root){
    if (!root || root.dataset.sim05Ep05Init) return;
    root.dataset.sim05Ep05Init = "1";

    var N = 4;
    var f = [[120, 130, 125, 128], [115, 140, 135, 122], [118, 150, 160, 130], [110, 120, 145, 138]];
    var Qbase = [[4, 6, 8, 10], [6, 8, 10, 12], [8, 10, 12, 16], [10, 12, 16, 20]];

    var s   = root.querySelector('#sim-ep0505_sl_s');
    var sv  = root.querySelector('#sim-ep0505_vl_s');
    var go  = root.querySelector('#sim-ep0505_grid_o');
    var gr  = root.querySelector('#sim-ep0505_grid_r');
    var dbg = root.querySelector('#sim-ep0505_debug');

    function alpha(k){ return k === 0 ? Math.sqrt(1 / N) : Math.sqrt(2 / N); }

    function dct2(g){
      var C = [];
      for (var u = 0; u < N; u++){ C.push(new Array(N).fill(0)); }
      for (var u = 0; u < N; u++){
        for (var v = 0; v < N; v++){
          var sum = 0;
          for (var x = 0; x < N; x++){
            for (var y = 0; y < N; y++){
              sum += g[x][y] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          C[u][v] = alpha(u) * alpha(v) * sum;
        }
      }
      return C;
    }

    function idct2(C){
      var g = [];
      for (var x = 0; x < N; x++){ g.push(new Array(N).fill(0)); }
      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          var sum = 0;
          for (var u = 0; u < N; u++){
            for (var v = 0; v < N; v++){
              sum += alpha(u) * alpha(v) * C[u][v] * Math.cos(Math.PI * (2 * x + 1) * u / (2 * N)) * Math.cos(Math.PI * (2 * y + 1) * v / (2 * N));
            }
          }
          g[x][y] = sum;
        }
      }
      return g;
    }

    function cell(v){
      var c = document.createElement('div');
      c.className = 'sim-ep0505_cell';
      c.style.cssText = 'background:#eafaf1; color:#04342C; border:1px solid #a3e4d7;';
      c.textContent = v;
      return c;
    }

    function render(){
      var scale = parseFloat(s.value);
      sv.innerHTML = scale.toFixed(2) + '&times;';
      go.innerHTML = '';
      gr.innerHTML = '';

      var g = [];
      for (var x = 0; x < N; x++){
        var row = [];
        for (var y = 0; y < N; y++){
          row.push(f[x][y] - 128);
        }
        g.push(row);
      }

      var C = dct2(g);
      var Cq = [];
      for (var u = 0; u < N; u++){
        var row = [];
        for (var v = 0; v < N; v++){
          var Qv = Qbase[u][v] * scale;
          var q = Math.round(C[u][v] / Qv);
          row.push(q * Qv);
        }
        Cq.push(row);
      }

      var gr2 = idct2(Cq);
      var diffSum = 0, n = 0;

      for (var x = 0; x < N; x++){
        for (var y = 0; y < N; y++){
          go.appendChild(cell(f[x][y]));
          var rec = Math.round(gr2[x][y] + 128);
          rec = Math.max(0, Math.min(255, rec));
          gr.appendChild(cell(rec));
          diffSum += Math.abs(rec - f[x][y]);
          n++;
        }
      }

      dbg.textContent = 'Mean absolute error per pixel: ' + (diffSum / n).toFixed(2) + '  |  Quanto maior a escala de Q, maior o erro de reconstrução.';
    }

    s.addEventListener('input', render);
    render();
  }

  function tryInitSim05Ep05(){
    var root = document.getElementById('sim-ep0505');
    if (root) initSim05Ep05(root); else setTimeout(tryInitSim05Ep05, 200);
  }
  tryInitSim05Ep05();
})();
</script>
""")

**Figure 5.5:** EP05_05 Simulator: Complete JPEG pipeline in block


In [ ]:
%%writefile EP05_05.py
# Python code

In [ ]:
TestSuite("EP05_05.py").run()